# 모델링 전처리

In [202]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import datetime

# 불필요한 경고문 생략(선택)
import warnings
warnings.filterwarnings('ignore')

# 모든 컬럼 출력설정(선택)
pd.set_option('display.max_columns', None)

#데이터 불러오기 
df = pd.read_csv('total_data.csv',index_col=0)

print('[행/컬럼 갯수]')
print(f"행: {df.shape[0]}, 컬럼: {df.shape[1]}\n")


[행/컬럼 갯수]
행: 51279, 컬럼: 40



In [203]:
#주차 컬럼 날짜타입 변환 (범주->날짜형)
df['주차'] = pd.to_datetime(df['주차'], format='%Y%m%d')
df['주차'].info()

<class 'pandas.core.series.Series'>
Index: 51279 entries, 0 to 51278
Series name: 주차
Non-Null Count  Dtype         
--------------  -----         
51279 non-null  datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 801.2 KB


In [204]:
# 결측치 확인 -> 없음 
df.isna().sum()

기간        0
주차        0
라인        0
성별        0
기획년도      0
시즌이월      0
상품년차      0
시즌        0
복종        0
소품종       0
CAT       0
총입고수량     0
총입고원가     0
총입고택가     0
총출고수량     0
총출고원가     0
총출고택가     0
판매액       0
판매수량      0
매출원가      0
판매택가      0
총판매액      0
총판매수량     0
총매출원가     0
총판매택가     0
물류재고수량    0
물류재고원가    0
물류재고택가    0
매장재고수량    0
매장재고원가    0
매장재고택가    0
재고수량      0
재고원가      0
재고택가      0
기간입고수량    0
기간입고원가    0
기간입고택가    0
기간출고수량    0
기간출고원가    0
기간출고택가    0
dtype: int64

In [205]:
# 중복값 확인 및 제거 -> 전체 중복 12개
df.duplicated().sum() 
df.drop_duplicates(inplace=True)
# df[df.duplicated(keep=False)].sort_values(by='주차')

print('[행/컬럼 갯수]')
print(f"행: {df.shape[0]}, 컬럼: {df.shape[1]}\n")

[행/컬럼 갯수]
행: 51267, 컬럼: 40



# 범주형 컬럼 확인

In [206]:
#성별 소품종 클래스 확인 : 기타로 분류되는 클래스 2개 존재
#--> 최종 '기타' 로 오분류된 항목 52개 변환
display(df['성별'].value_counts())

sex_df = df[df['성별'].str.contains('기타')]
sex_df['소품종'].value_counts()

#봄 패딩 베스트만, '기타' 로 분류됨 -> 성별 구분 착오 예상 --> 봄패딩베스트 '1:남성' 값으로 변환 
con = (df['소품종']=='패딩베스트') & (df['시즌']=='봄')
df.loc[con,'성별'] = '1:남성'

df.loc[con,'성별'].value_counts()

성별
1:남성      47346
3:남녀공용     2510
2:여성       1322
4:기타         89
Name: count, dtype: int64

성별
1:남성    52
Name: count, dtype: int64

In [207]:
#시즌이월 컬럼 클래스 확인 : 이월제품 의미 파악 필요
##결론 : 판매예측/할인최적화 모델링시 '이월' 행 삭제 (-12,578 32%) ,년간 매출 집계시 유지
display(df['시즌이월'].value_counts())

#2024년도 기준 시즌/이월 여부 확인 
df_2024 = df[df['기획년도']==2024]
df_2024 = df_2024.drop(columns=['기간','상품년차'])

# # 범주형 최소단위 필터링을 위한 '카테고리' 컬럼 생성
df_2024['카테고리'] = df_2024['시즌'] + "_" +  df_2024['복종'] + "_" + df_2024['소품종'] + "_" + df_2024['라인']+ "_" + df_2024['성별']

# #카테고리별 시즌/이월값이 둘다 있는거 소팅 -> '샘플 가을_니트 셔츠_라운드_ZB_1:남성'   확인
df_2024.groupby('카테고리')['시즌이월'].nunique()

# #샘플확인 : '가을_니트 셔츠_라운드_ZB_1:남성' -> 시즌별 마감 이후 이월로 변경 됨 
con = df_2024['카테고리'] == '가을_니트 셔츠_라운드_ZB_1:남성'
smpl = df_2024[con].sort_values(by='주차',ascending=True)
smpl[(smpl['주차'] >'2024-11-01') & (smpl['주차'] <='2024-12-30')]

# 시즌-> 이월 바뀌는 시점 함수화 : gpt
def find_transition_points(group):
    group = group.sort_values('주차')
    transition_rows = group[(group['시즌이월'].shift(1) == '01_시즌') & (group['시즌이월'] == '02_이월')]
    return transition_rows[['카테고리', '주차']]

transition_points = df_2024.groupby('카테고리', group_keys=False).apply(find_transition_points)

# 시즌 -> 이월로 바뀌는 주차만 출력
transition_points['주차'].unique()

시즌이월
01_시즌    38689
02_이월    12578
Name: count, dtype: int64

<DatetimeArray>
['2024-12-01 00:00:00', '2024-06-02 00:00:00', '2024-10-06 00:00:00']
Length: 3, dtype: datetime64[ns]

In [208]:
print('[행/컬럼 갯수]')
print(f"행: {df.shape[0]}, 컬럼: {df.shape[1]}\n")

df.head(3)

[행/컬럼 갯수]
행: 51267, 컬럼: 40



,기간,주차,라인,성별,기획년도,시즌이월,상품년차,시즌,복종,소품종,CAT,총입고수량,총입고원가,총입고택가,총출고수량,총출고원가,총출고택가,판매액,판매수량,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가,물류재고수량,물류재고원가,물류재고택가,매장재고수량,매장재고원가,매장재고택가,재고수량,재고원가,재고택가,기간입고수량,기간입고원가,기간입고택가,기간출고수량,기간출고원가,기간출고택가
0,당해,2021-01-03,ZB,1:남성,2021,01_시즌,당해년도,봄,우븐 셔츠,캐쥬얼셔츠,02_SHIRTS,2247,19893640,157065300,1009,8933103,70529100,-124850,0,0,0,84850,3,26560,209700,1238,10960537,86536200,1006,8906543,70319400,2244,19867080,156855600,0,0,0,53,469228,3704700
1,당해,2021-01-03,ZB,1:남성,2021,01_시즌,당해년도,사계절,소품,양말,10_ACC/ETC,14000,11480000,46200000,10775,8835500,35557500,2033156,617,505940,2036100,10270838,3127,2564140,10319100,3225,2644500,10642500,7648,6271360,25238400,10873,8915860,35880900,0,0,0,454,372280,1498200
2,당해,2021-01-03,ZB,1:남성,2021,01_시즌,당해년도,봄,니트 셔츠,라운드,01_KNIT,20076,178239296,1403312400,5940,52736683,415206000,18981424,401,3559559,28029900,18981424,401,3560170,28029900,14136,125502613,988106400,5539,49176513,387176100,19675,174679125,1375282500,0,0,0,5940,52713656,415206000


# 수치형 컬럼 & 집계 컬럼 점검
사용 컬럼 : '총입고수량','총입고원가','총입고택가','총출고수량','총출고원가','총출고택가','판매수량','판매액','매출원가','판매택가','총판매액','총판매수량','총매출원가','총판매택가','재고수량','재고원가','재고택가'

In [209]:
# 총입고수량/입고원가/입고택가  -> 전처리 (-162행)
# 입고전 데이터 확인 및 행 삭제 : 162개 -> 입고되지 않은 상품은 출고 및 판매 불가, 예약판매 등 특수한 케이스 없다고 가정 

#1. 범주형 최소단위 필터링을 위한 '카테고리' 컬럼 생성
df['카테고리'] = df['시즌'] + "_" +  df['복종'] + "_" + df['소품종'] + "_" + df['라인']
num_df = df[['카테고리','주차','총입고수량','총입고원가','총입고택가','판매수량','판매액','매출원가','판매택가','총판매액','총판매수량','총매출원가','총판매택가','재고수량','재고원가','재고택가','시즌','복종','소품종','라인','시즌이월','기획년도']]

print((num_df[f'총입고수량'] == 0).sum())
filtered_df = num_df[(df['총입고수량'] > 0)]

print('[행/컬럼 갯수]')
print(f"행: {filtered_df.shape[0]}, 컬럼: {filtered_df.shape[1]}\n")

162
[행/컬럼 갯수]
행: 51105, 컬럼: 22



In [210]:
# 재고수량/재고원가/재고택가 정합성 확인 -> 재고 관련 컬럼 삭제 
# 결론: 재고관련 집계 컬럼 삭제 후 입고-판매 기준 다시 집계 (입고,판매 데이터의 신뢰도가 더 높다고 봄, 실제 wms랑 비교 할 수 없으므로 가정)
filtered_df['재고잔량_check'] = (filtered_df['총입고수량'] - filtered_df['총판매수량'] == filtered_df['재고수량'])
filtered_df['재고원가_check'] = (filtered_df['총입고원가'] - filtered_df['총매출원가'] == filtered_df['재고원가'])
filtered_df['재고택가_check'] = (filtered_df['총입고택가'] - filtered_df['총판매택가'] == filtered_df['재고원가'])

# 입고 - 판매 = 재고 안맞는 행 : 27569행
filtered_df[filtered_df[['재고잔량_check', '재고원가_check', '재고택가_check']].any(axis=1) == False]
(filtered_df[['재고잔량_check', '재고원가_check', '재고택가_check']].any(axis=1) == False).sum() #27569행

# 재고관련 컬럼 삭제
filtered_df.drop(columns=['재고수량','재고원가','재고택가','재고잔량_check','재고원가_check','재고택가_check'],inplace=True)

print('[행/컬럼 갯수]')
print(f"행: {filtered_df.shape[0]}, 컬럼: {filtered_df.shape[1]}\n")

[행/컬럼 갯수]
행: 51105, 컬럼: 19



# 모델링 전처리 진행 

In [211]:
# 모델링용 추출시 아래 내용 삭제 
# 특정 라인(ZD, ZE, ZF) 제거
filtered_df = filtered_df[~filtered_df['라인'].isin(['ZD', 'ZE', 'ZF'])]

# '소품', '언더웨어' 제거
filtered_df = filtered_df[~filtered_df['복종'].isin(['소품', '언더웨어'])]

# # 시즌이월 '이월' 제거
filtered_df = filtered_df[~filtered_df['시즌이월'].isin(['02_이월'])]

print('[행/컬럼 갯수]')
print(f"행: {filtered_df.shape[0]}, 컬럼: {filtered_df.shape[1]}\n")
filtered_df.head(3)

[행/컬럼 갯수]
행: 24074, 컬럼: 19



,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가,시즌,복종,소품종,라인,시즌이월,기획년도
0,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-03,2247,19893640,157065300,0,-124850,0,0,84850,3,26560,209700,봄,우븐 셔츠,캐쥬얼셔츠,ZB,01_시즌,2021
2,봄_니트 셔츠_라운드_ZB,2021-01-03,20076,178239296,1403312400,401,18981424,3559559,28029900,18981424,401,3560170,28029900,봄,니트 셔츠,라운드,ZB,01_시즌,2021
4,봄_수트_블레이져(수트)_ZA,2021-01-03,405,26409229,153495000,6,1682760,391248,2274000,16797003,60,3912478,22740000,봄,수트,블레이져(수트),ZA,01_시즌,2021


In [212]:
# 그룹화 및 집계 연산 적용 :최종 컬럼 8867행
group_cols = ['시즌','복종','소품종','라인','시즌이월','카테고리','기획년도','주차']
agg_dict = {
    '총입고수량': 'sum',
    '총입고원가': 'sum',
    '총입고택가': 'sum',

    '판매수량': 'sum',
    '판매액': 'sum',
    '매출원가': 'sum',
    '판매택가': 'sum'
}

filtered_df = filtered_df.groupby(group_cols).agg(agg_dict).reset_index()

print('[행/컬럼 갯수]')
print(f"행: {filtered_df.shape[0]}, 컬럼: {filtered_df.shape[1]}\n")

[행/컬럼 갯수]
행: 8867, 컬럼: 15



- 파생변수 생성 

In [213]:
# 1. 입고기준 원가/택가 -> 정수로 반올림
filtered_df['제품원가'] = (filtered_df['총입고원가'] / filtered_df['총입고수량']).round(0)
filtered_df['제품택가'] = (filtered_df['총입고택가'] / filtered_df['총입고수량']).round(0)

df['카테고리'].nunique()

252

In [214]:
# 2. 카테고리 시즌별 평균 판매수량, 평균 실판가 집계 (아래 고려사항)
# 카테고리별 평균 판매수량
# 카테고리별 평균 실판가
# 시즌내 마감일자 기간 까지의 평균 값 적용 
# 정수까지 반올림

print('[최초 행/컬럼 갯수]')
print(f"행: {filtered_df.shape[0]}, 컬럼: {filtered_df.shape[1]}\n")

avg_df= filtered_df
avg_df['평균실판가'] = avg_df['판매액'] / avg_df['판매수량']

avg_df2 =avg_df.groupby(['카테고리','기획년도'])[['판매수량','평균실판가']].mean().reset_index()
avg_df2 = avg_df2.round(0) #평균값 
avg_df2

[최초 행/컬럼 갯수]
행: 8867, 컬럼: 17



,카테고리,기획년도,판매수량,평균실판가
0,가을_니트 셔츠_라운드_ZB,2021,869.0,35654.0
1,가을_니트 셔츠_라운드_ZB,2022,202.0,40876.0
2,가을_니트 셔츠_라운드_ZB,2023,227.0,34667.0
3,가을_니트 셔츠_라운드_ZB,2024,197.0,19876.0
4,가을_니트 셔츠_티에리_ZB,2024,148.0,28098.0
...,...,...,...,...
346,여름_팬츠_팬츠(일반)_ZB,2021,3677.0,36407.0
347,여름_팬츠_팬츠(일반)_ZB,2022,3079.0,34232.0
348,여름_팬츠_팬츠(일반)_ZB,2023,2447.0,44048.0
349,여름_팬츠_팬츠(일반)_ZB,2024,2082.0,43988.0


In [215]:
# 각 데이터프레임의 카테고리 개수 확인
filtered_count = filtered_df['카테고리'].nunique()
avg_count = avg_df2['카테고리'].nunique()

print(f"최초 카테고리 개수: {filtered_count}")
print(f"avg 카테고리 개수: {avg_count}")

#본 데이터에 평균실판가/평균판매수량 병합
merge_df = filtered_df.merge(avg_df2, on=['카테고리','기획년도'], how='inner') # **기획년도추가 

merge_df = merge_df.rename(columns={'판매수량_x' : '판매수량','판매수량_y' : '평균판매수량'})
merge_df.head(3)

#최종 merge_df

최초 카테고리 개수: 133
avg 카테고리 개수: 133


,시즌,복종,소품종,라인,시즌이월,카테고리,기획년도,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,제품원가,제품택가,평균실판가_x,평균판매수량,평균실판가_y
0,가을,니트 셔츠,라운드,ZB,01_시즌,가을_니트 셔츠_라운드_ZB,2021,2021-07-11,10931,90858472,764076900,0,0,0,0,8312.0,69900.0,NaN,869.0,35654.0
1,가을,니트 셔츠,라운드,ZB,01_시즌,가을_니트 셔츠_라운드_ZB,2021,2021-07-18,10931,90858472,764076900,56,3131520,466312,3914400,8312.0,69900.0,55920.000000,869.0,35654.0
2,가을,니트 셔츠,라운드,ZB,01_시즌,가을_니트 셔츠_라운드_ZB,2021,2021-07-25,10931,90858472,764076900,-7,-506220,-59024,-489300,8312.0,69900.0,72317.142857,869.0,35654.0


In [216]:
# 음수 데이터 확인 -> 판매수량/판매액/매출원가/판매택가의 음수값 갯수가 다 다름
minus_con = merge_df.select_dtypes(include='number') <0
minus_con.sum()

기획년도         0
총입고수량        0
총입고원가        0
총입고택가        0
판매수량       121
판매액        153
매출원가       129
판매택가       124
제품원가         0
제품택가         0
평균실판가_x     38
평균판매수량       0
평균실판가_y      0
dtype: int64

In [217]:
# 1. 판매수량은 0인데, 판매액/매출원가/판매택가가 있는 경우
#결론: 판매 없이 판매액/매출원가/판매택가 발생할수없다고 판단, 계산오류 간주하고 매출원가/판매택가 0으로 변경
con1 = merge_df['판매수량'] == 0
con2 = merge_df['판매액'] != 0
con3 = merge_df['매출원가'] != 0
con4 = merge_df['판매택가'] != 0

print((con1 & (con2 | con3 | con4)).sum())

#수량이 0일떄, 판매액/매출원가 0으로 변경
merge_df.loc[(con1 & (con2 | con3 | con4)),['판매수량','판매액','매출원가']] = 0

merge_df.describe()

8


,기획년도,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,제품원가,제품택가,평균실판가_x,평균판매수량,평균실판가_y
count,8867.000000,8867,8867.000000,8.867000e+03,8.867000e+03,8867.000000,8.867000e+03,8.867000e+03,8.867000e+03,8867.000000,8.867000e+03,8.585000e+03,8867.000000,8683.000000
mean,2022.557911,2023-02-01 16:52:43.550242304,24190.836247,3.478319e+08,2.763200e+09,552.043532,3.202081e+07,8.662314e+06,6.820168e+07,24379.986354,1.686003e+05,-inf,552.026728,92535.976736
min,2021.000000,2021-01-03 00:00:00,352.000000,5.031646e+06,3.990000e+07,-1821.000000,-1.703995e+08,-3.288194e+07,-2.504720e+08,3853.000000,1.990000e+04,-inf,7.000000,14508.000000
25%,2022.000000,2022-02-13 00:00:00,5282.000000,1.058318e+08,8.059580e+08,64.000000,4.725150e+06,1.212350e+06,9.211000e+06,10292.000000,9.079100e+04,3.923588e+04,125.000000,41432.000000
50%,2023.000000,2023-02-19 00:00:00,10616.000000,2.222915e+08,1.569115e+09,203.000000,1.600555e+07,4.139393e+06,3.107700e+07,17665.000000,1.290000e+05,7.233333e+04,293.000000,74139.000000
75%,2024.000000,2024-02-04 00:00:00,23510.000000,4.538735e+08,3.176920e+09,559.500000,4.216219e+07,1.070410e+07,8.223450e+07,31591.000000,2.390000e+05,1.251951e+05,567.000000,120372.000000
max,2024.000000,2024-12-29 00:00:00,739995.000000,3.281674e+09,3.883962e+10,38139.000000,6.324048e+08,2.658176e+08,2.305161e+09,266866.000000,1.199000e+06,2.705800e+06,12996.000000,639534.000000
std,1.109350,NaN,51047.244277,3.836101e+08,3.570205e+09,1373.979712,4.515187e+07,1.387824e+07,1.230758e+08,23776.409783,1.242007e+05,NaN,1025.423419,73478.041900


In [218]:
# 2.판매수량/ 매출원가 부호정합성 (제품원가 * 판매수량)
# 판매수량 >0 , 집계컬럼 <=0 (매출원가)
# 결론 :집계 오류 판단 -> 입고원가 * 판매수량 으로 대치
con = (merge_df['판매수량'] >= 0) & (merge_df['매출원가'] < 0)
display(con.sum())

merge_df.loc[con,'매출원가']= merge_df['제품원가'] * merge_df['판매수량'] 
merge_df[con]

# 판매수량 <0 , 집계컬럼 >=0 (매출원가)
con = (merge_df['판매수량'] < 0) & (merge_df['매출원가'] >= 0)
display(con.sum())

merge_df.loc[con,'매출원가']= merge_df['제품원가']* merge_df['판매수량'] 
merge_df[con]

5

0

,시즌,복종,소품종,라인,시즌이월,카테고리,기획년도,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,제품원가,제품택가,평균실판가_x,평균판매수량,평균실판가_y


In [219]:
# 3.판매수량/ 판매액 부호정합성 -> (평균판매액 * 판매수량)
#판매수량 >0 , 집계컬럼 <0 (판매액)
#총 28행 
con1 = (merge_df['판매수량'] > 0) & (merge_df['판매액'] < 0)
display(con1.sum())

#판매수량 <0, 집계컬럼 >0 (판매액)
#총 3행 
con2 = (merge_df['판매수량'] < 0) & (merge_df['판매액'] > 0)
display(con2.sum())

#판매수량 !=0, 집계컬럼 ==0 (판매액) 
#없음
con3 = (merge_df['판매수량'] != 0) & (merge_df['판매액'] == 0)
display(con3.sum())
merge_df[con3]

# 총 이상치 갯수
(con1 | con2 | con3).sum()

merge_df.loc[(con1 | con2 | con3),'판매액'] = merge_df['판매수량'] * merge_df['평균실판가'] 
merge_df

28

3

0

KeyError: '평균실판가'

In [ ]:
# 4.판매수량/ 판매택가 부호정합성 (제품택가 * 판매수량)
# 판매수량 >0 , 집계컬럼 <=0 (판매택가)
# 결론 : 3행 존재 집계 오류 판단 -> 입고택가 * 판매수량 으로 대치
con = (merge_df['판매수량'] >= 0) & (merge_df['판매택가'] < 0)
display(con.sum())

merge_df.loc[con,'판매택가']= merge_df['제품택가'] * merge_df['판매수량'] 
merge_df[con]

# 판매수량 <0 , 집계컬럼 >=0 (판매택가)
# 결론 : 없음
con1 = (merge_df['판매수량'] < 0) & (merge_df['판매택가'] >= 0)
display(con1.sum())

merge_df[con]

3

0

,시즌,복종,소품종,라인,시즌이월,카테고리,기획년도,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,제품원가,제품택가,평균판매수량,평균실판가
2183,겨울,자켓,싱글재킷,ZA,01_시즌,겨울_자켓_싱글재킷_ZA,2024,2024-12-29,10296,431502948,3379704000,16,NaN,670560,5252064,41910.0,328254.0,248.0,NaN
2316,겨울,점퍼,패딩점퍼,ZB,01_시즌,겨울_점퍼_패딩점퍼_ZB,2024,2024-09-15,8030,254473597,1747970000,0,0.0,0,0,31690.0,217680.0,422.0,NaN
3999,봄,코트,싱글코트,ZB,01_시즌,봄_코트_싱글코트_ZB,2022,2022-01-16,4477,112452938,1347893000,0,0.0,0,0,25118.0,301071.0,371.0,134681.0


In [ ]:
# 음수 데이터 확인 -> 판매수량/판매액/매출원가/판매택가의 음수값 갯수가 다 다름
minus_con = merge_df.select_dtypes(include='number') <0
minus_con.sum()

기획년도        0
총입고수량       0
총입고원가       0
총입고택가       0
판매수량      121
판매액       121
매출원가      121
판매택가      121
제품원가        0
제품택가        0
평균판매수량      0
평균실판가       0
dtype: int64

# 파생변수 추가
- 주차별 실판가/할인율 
- 누적판매수량 / 누적판매액 / 누적매출원가 /누적판매택가
- 누적판매율(수량) / roi 
** 정수단위 반올림 / roi 는 소수점 둘째까지 반올림

In [ ]:
#제품실판가,할인율
merge_df['제품실판가'] = merge_df['판매액']/merge_df['판매수량'] #null 값 있음
merge_df['할인율(%)'] = (merge_df['판매택가'] - merge_df['판매액']) / merge_df['판매택가']*100 #null 값 있음

#누적판매데이터: cumsum()
merge_df = merge_df.sort_values(by=['카테고리','주차'],ascending=True)

merge_df['누적판매수량'] = merge_df.groupby(['기획년도','카테고리'])['판매수량'].cumsum()
merge_df['누적판매액'] = merge_df.groupby(['기획년도','카테고리'])['판매액'].cumsum()
merge_df['누적매출원가'] = merge_df.groupby(['기획년도','카테고리'])['매출원가'].cumsum()
merge_df['누적판매택가'] = merge_df.groupby(['기획년도','카테고리'])['판매택가'].cumsum()

#판매율 / roi
merge_df['누적판매율(%)'] = merge_df['누적판매수량']/merge_df['총입고수량']*100
merge_df['ROI'] = ((merge_df['누적판매액']/1.1 - merge_df['누적매출원가'])/merge_df['총입고원가']).round(2) #roi 소수점 2자리 까지

# 실수형 서식 변환 : 반올림0까지
round_cols = ['제품실판가', '할인율(%)','누적판매율(%)']
merge_df[round_cols] = merge_df[round_cols].round(0)

print('[행/컬럼 갯수]')
print(f"행: {merge_df.shape[0]}, 컬럼: {merge_df.shape[1]}\n")
model_final_df = merge_df 

[행/컬럼 갯수]


행: 8867, 컬럼: 27



In [ ]:
# csv 파일 추출 
# model_final_df.to_csv('모델링전처리.csv')

# 최종 전처리 df : model_final_df

# 판매 음수값 이상치 확인 및 처리 -> 모델링시 필요
- 1. 1.5 iqr
- 2. 3시그마
- 3. 매장기준 -200 절대값


In [ ]:
# 판매수량이 음수인 값만 필터링
negative_sales_df = model_final_df[model_final_df['판매수량'] < 0]

# IQR 계산
Q1 = negative_sales_df['판매수량'].quantile(0.25)
Q3 = negative_sales_df['판매수량'].quantile(0.75)
IQR = Q3 - Q1

# 3 * IQR 기준으로 이상치 판별
lower_bound = Q1 - 1.5 * IQR

# 이상치 개수 확인
outliers_count = (negative_sales_df['판매수량'] < lower_bound).sum()

# 결과 출력
print(f"이상치 기점: {lower_bound}, 3 IQR 이상인 이상치 개수: {outliers_count}")

이상치 기점: -246.0, 3 IQR 이상인 이상치 개수: 20


In [ ]:
# 3표준편차 -> 하한 지점이 너무 낮음 -> 탈락 
mean = negative_sales_df['판매수량'].mean()
stddev = negative_sales_df['판매수량'].std()
lower_bound2 = mean - 3 * stddev  # 3표준편차 하한

# 이상치 개수 확인
outliers_count2 = (negative_sales_df['판매수량'] < lower_bound2).sum()

# 결과 출력
print(f"이상치 기점: {lower_bound2}, 3시그마 이상인 이상치 개수: {outliers_count2}")

이상치 기점: -1157.0053274530037, 3시그마 이상인 이상치 개수: 4


In [ ]:
# 전국 매장 수 기준 : 약 200개
# 이상치 개수 확인
outliers_count3 = (negative_sales_df['판매수량'] < -200).sum()

# 결과 출력
print(f"이상치 기점: -200, 3 IQR 이상인 이상치 개수: {outliers_count3}")

이상치 기점: -200, 3 IQR 이상인 이상치 개수: 21


In [ ]:
# 카테고리별 개수 세기 - 1.5 iqr 기준
category_counts = negative_sales_df[negative_sales_df['판매수량'] < lower_bound]
category_outlier_counts = category_counts['카테고리'].value_counts()
category_outlier_counts

카테고리
사계절_니트 셔츠_라운드_ZB      7
사계절_데님_데님팬츠_ZB        6
여름_니트 셔츠_라운드_ZB       2
여름_자켓_싱글재킷_ZB         2
겨울_스웨터_라운드_ZB         1
봄_자켓_싱글재킷_ZB          1
사계절_우븐 셔츠_드레스셔츠_ZB    1
Name: count, dtype: int64

# 음수 판매량 중 이상치 값 어디서 / 얼마나 뺄것인가? 즉, 가매출 정상화 로직
#1. 종한ver : 음수 이상치를 평균판매수량을 변경한 후 가매출 시점에 (음수판매수량)-(평균판매수량) 을 합집계
- 평균 수량 : 80
- 현 주차 판매수량: -200
- 가 매출주차 수량 : 500 이라면 ,-(200 + 80) => 220 
#결과 : 입고수량 유지하면서 현주차 판매수량 평균으로 대체 : 500+(-200) = 220+80

#2. 최종 채택 ver : 음수이상치 0으로 변경한 후, 해당수량을 가매출 시점에서 빼줌 